In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the cleaned sleep dataset
df = pd.read_csv('../data/processed/sleep_cleansed.csv')

# 2. Separate Features (X) and Target (y)
X = df.drop(columns=['Target_Sleep_Disorder'])
y = df['Target_Sleep_Disorder']

# 3. Stratified Train/Test Split (80/20)
# Stratify ensures the None, Insomnia, and Sleep Apnea classes are balanced in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Data Split Complete ---")
print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")

--- Data Split Complete ---
Training Features Shape: (299, 12)
Testing Features Shape: (75, 12)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. Define which columns need which preprocessing
categorical_cols = ['Gender', 'Occupation', 'BMI Category']
numerical_cols = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 
                  'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic_BP', 'Diastolic_BP']

# 2. Build the Preprocessor
# StandardScaler ensures large numbers (Daily Steps) don't overpower small numbers (Sleep Duration)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 3. Build and Train the Pipeline with Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

# 4. Predict and Evaluate on the unseen Test Set
y_pred_lr = lr_pipeline.predict(X_test)

print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred_lr, average='macro'):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['None', 'Insomnia', 'Sleep Apnea']))

--- Logistic Regression Results ---
Accuracy: 0.8933
Macro F1: 0.8663

Classification Report:
              precision    recall  f1-score   support

        None       1.00      0.91      0.95        44
    Insomnia       0.92      0.80      0.86        15
 Sleep Apnea       0.68      0.94      0.79        16

    accuracy                           0.89        75
   macro avg       0.87      0.88      0.87        75
weighted avg       0.92      0.89      0.90        75



In [3]:
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# 1. XGBoost strictly requires targets to be numeric (0, 1, 2)
# We will use LabelEncoder to ensure y is properly formatted for all models
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Define our advanced models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss')
}

best_model_name = "Logistic Regression"
best_macro_f1 = f1_score(y_test, y_pred_lr, average='macro')
best_pipeline = lr_pipeline # from the previous cell

print("--- Advanced Models Evaluation ---\n")

# 3. Train and compare
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train the model
    pipeline.fit(X_train, y_train_enc)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test_enc, y_pred)
    macro_f1 = f1_score(y_test_enc, y_pred, average='macro')
    
    print(f"{name}:")
    print(f"Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}\n")
    
    # Check if this model beats our current best
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_model_name = name
        best_pipeline = pipeline

print(f"🏆 Overall Best Model: {best_model_name} (Macro F1: {best_macro_f1:.4f})")

# 4. Save the winning pipeline for Streamlit deployment
os.makedirs("../models", exist_ok=True)
model_path = os.path.join("..", "models", "sleep_disorder_model.pkl")
joblib.dump(best_pipeline, model_path)
print(f"✅ Winning model successfully saved to: {model_path}")

--- Advanced Models Evaluation ---

Random Forest:
Accuracy: 0.9333 | Macro F1: 0.9009

XGBoost:
Accuracy: 0.9067 | Macro F1: 0.8588

🏆 Overall Best Model: Random Forest (Macro F1: 0.9009)
✅ Winning model successfully saved to: ..\models\sleep_disorder_model.pkl
